<a href="https://colab.research.google.com/github/andreelzs/Linguagens-de-programacao/blob/main/analise_vendas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Vendas - E-commerce

In [1]:
import numpy as np
import pandas as pd

dados_vendas = {
    'cliente_id': [101, 102, 103, 101, 104, 102, 105, 103],
    'valor': [3500.75, 189.50, np.nan, 1200.00, 450.00, np.nan, 89.90, 780.50],
    'categoria': ['Eletronicos', 'Livros', 'Roupas', 'Eletronicos', 'Automotivo', 'Livros', 'Roupas', 'Roupas'],
    'data_hora': ['2024-01-15 10:23:00', '2024-01-18 14:05:00', '2024-02-05 09:12:00',
                  '2024-02-20 16:40:00', '2024-03-02 11:00:00', '2024-03-15 18:30:00',
                  '2024-04-10 08:20:00', '2024-04-22 13:45:00'],
    'status': ['Concluído', 'Concluído', 'Pendente', 'Concluído', 'Cancelado', 'Concluído', 'Concluído', 'Concluído'],
    'email': ['maria@gmail.com', 'joao@outlook.com', 'ana@yahoo.com', 'maria@gmail.com',
              'carlos@gmail.com', 'joao@outlook.com', 'lucas@empresa.com.br', 'ana@yahoo.com'],
}

dados_clientes = {
    'cliente_id': [101, 102, 103, 104, 105],
    'nome': ['Maria Silva', 'Joao Souza', 'Ana Oliveira', 'Carlos Lima', 'Lucas Mendes'],
    'cidade': ['Sao Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Curitiba', 'Salvador'],
}

df_vendas = pd.DataFrame(dados_vendas)
df_clientes = pd.DataFrame(dados_clientes)

## Parte 1: Diagnóstico e Limpeza

In [2]:
# 1. Dimensões e tipos
print(df_vendas.shape)
df_vendas.info()

(8, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   cliente_id  8 non-null      int64  
 1   valor       6 non-null      float64
 2   categoria   8 non-null      object 
 3   data_hora   8 non-null      object 
 4   status      8 non-null      object 
 5   email       8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [3]:
# 2. Nulos e preenchimento pela mediana da categoria
print(df_vendas.isnull().sum())

mediana_categoria = df_vendas.groupby('categoria')['valor'].transform('median')
df_vendas['valor'] = df_vendas['valor'].fillna(mediana_categoria)
df_vendas['valor'] = df_vendas['valor'].fillna(df_vendas['valor'].median())
df_vendas

cliente_id    0
valor         2
categoria     0
data_hora     0
status        0
email         0
dtype: int64


,cliente_id,valor,categoria,data_hora,status,email
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com


In [4]:
# 3. Concluídos com valor > 500
df_vendas.query("status == 'Concluído' and valor > 500")

,cliente_id,valor,categoria,data_hora,status,email
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com


## Parte 2: Cruzamento e Transformação

In [5]:
# 4. Left merge
df_merged = df_vendas.merge(df_clientes, on='cliente_id', how='left')
df_merged

,cliente_id,valor,categoria,data_hora,status,email,nome,cidade
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com,Ana Oliveira,Belo Horizonte
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com,Carlos Lima,Curitiba
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br,Lucas Mendes,Salvador
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com,Ana Oliveira,Belo Horizonte


In [6]:
# 5. Média por categoria sem reduzir linhas
df_merged['media_categoria'] = df_merged.groupby('categoria')['valor'].transform('mean')
df_merged

,cliente_id,valor,categoria,data_hora,status,email,nome,cidade,media_categoria
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo,2350.375
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro,189.500
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com,Ana Oliveira,Belo Horizonte,435.200
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo,2350.375
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com,Carlos Lima,Curitiba,450.000
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro,189.500
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br,Lucas Mendes,Salvador,435.200
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com,Ana Oliveira,Belo Horizonte,435.200


In [7]:
# 6. Clientes com @gmail.com
qtd_gmail = df_merged['email'].str.contains('@gmail.com').sum()
clientes_gmail = df_merged.loc[df_merged['email'].str.contains('@gmail.com'), 'nome'].unique()
print(qtd_gmail, clientes_gmail)

3 ['Maria Silva' 'Carlos Lima']


## Parte 3: Análise Temporal e Agregação

In [8]:
# 7. Conversão datetime + mês/dia da semana
df_merged['data_hora'] = pd.to_datetime(df_merged['data_hora'])
df_merged['mes'] = df_merged['data_hora'].dt.month_name()
df_merged['dia_semana'] = df_merged['data_hora'].dt.day_name()
df_merged

,cliente_id,valor,categoria,data_hora,status,email,nome,cidade,media_categoria,mes,dia_semana
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo,2350.375,January,Monday
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro,189.500,January,Thursday
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com,Ana Oliveira,Belo Horizonte,435.200,February,Monday
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo,2350.375,February,Tuesday
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com,Carlos Lima,Curitiba,450.000,March,Saturday
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro,189.500,March,Friday
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br,Lucas Mendes,Salvador,435.200,April,Wednesday
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com,Ana Oliveira,Belo Horizonte,435.200,April,Monday


In [9]:
# 8. Pivot table: categoria x cidade, soma de valor
pivot = pd.pivot_table(df_merged, values='valor', index='categoria', columns='cidade',
                        aggfunc='sum', fill_value=0)
pivot

cidade,Belo Horizonte,Curitiba,Rio de Janeiro,Salvador,Sao Paulo
categoria,,,,,
Automotivo,0.0,450.0,0.0,0.0,0.00
Eletronicos,0.0,0.0,0.0,0.0,4700.75
Livros,0.0,0.0,379.0,0.0,0.00
Roupas,1215.7,0.0,0.0,89.9,0.00
